In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/ventas_limpio.csv')
print(f"Filas: {df.shape[0]}, Columnas: {df.shape[1]}")
df.head()

Filas: 5993, Columnas: 22


,id_venta,numero_linea,fecha_venta,id_cliente,genero_cliente,tipo_cliente,id_producto,nombre_producto,categoria,tipo_prenda,...,ciudad,departamento,empresa,cantidad,precio_unitario,descuento,metodo_pago,canal,temporada,talla
0,V003565,1,2024-02-04,C00909,Hombre,Local,P0003,Jean Mom,Jeans,Pantalón,...,Neiva,Huila,KOAJ,4,179000.0,0.0,Tarjeta Crédito,Tienda Física,Temporada Regular,40
1,V002111,2,2023-01-01,C00348,Hombre,Local,P0008,Chaqueta Bomber,Chaquetas,Chaqueta,...,Bogotá,Bogotá D.C.,KOAJ,1,211000.0,0.0,Efectivo,Tienda Física,Regreso A Clases,M
2,V002185,1,2023-03-31,C00804,Mujer,Turista,P0009,Camisa Casual,Camisas,Camisa,...,Santa Marta,Magdalena,KOAJ,1,128000.0,0.0,Efectivo,Tienda Física,Día Del Padre,Xxl
3,V002828,1,2024-05-13,C00951,Hombre,Local,P0014,Jogger,Pantalones,Pantalón,...,Barranquilla,Atlántico,KOAJ,2,133000.0,0.0,Tarjeta Crédito,Tienda Física,Black Friday,38
4,V001693,1,2023-11-14,C01804,Mujer,Local,P0007,Gorra,Accesorios,Accesorio,...,Bucaramanga,Santander,KOAJ,1,61000.0,0.0,Tarjeta Crédito,Tienda Física,Temporada Regular,Xl


In [4]:
# Crear tabla de dimensiones de productos

dim_producto = df[['id_producto', 'nombre_producto', 'categoria', 'tipo_prenda']].copy()

# Priorizar valores reales sobre 'Sin nombre' / 'Sin categoría'
cols_texto = ['nombre_producto', 'categoria', 'tipo_prenda']
for col in cols_texto:
    dim_producto[f'_sin_{col}'] = dim_producto[col].str.startswith('Sin').astype(int)

dim_producto = dim_producto.sort_values(['id_producto'] + [f'_sin_{c}' for c in cols_texto])
dim_producto = dim_producto.drop_duplicates(subset=['id_producto'], keep='first')
dim_producto = dim_producto.drop(columns=[f'_sin_{c}' for c in cols_texto])
dim_producto = dim_producto.sort_values('id_producto').reset_index(drop=True)

dim_producto


,id_producto,nombre_producto,categoria,tipo_prenda
0,P0001,Pantalón Clásico,Pantalones,Pantalón
1,P0002,Set Deportivo,Deportivo,Conjunto
2,P0003,Jean Mom,Jeans,Pantalón
3,P0004,Jean Skinny,Jeans,Pantalón
4,P0005,Falda,Faldas,Falda
5,P0006,Cinturón,Accesorios,Accesorio
6,P0007,Gorra,Accesorios,Accesorio
7,P0008,Chaqueta Bomber,Chaquetas,Chaqueta
8,P0009,Camisa Casual,Camisas,Camisa
9,P0010,Short,Shorts,Short


In [5]:
# Crear tabla de dimensiones de clientes

dim_cliente = df[['id_cliente', 'genero_cliente', 'tipo_cliente']].copy()

dim_cliente = dim_cliente.sort_values('genero_cliente')
dim_cliente = dim_cliente.drop_duplicates(subset=['id_cliente'], keep='first')
dim_cliente = dim_cliente.sort_values('id_cliente').reset_index(drop=True)

dim_cliente

,id_cliente,genero_cliente,tipo_cliente
0,C00001,Hombre,Turista
1,C00003,Hombre,Local
2,C00004,Mujer,Local
3,C00006,Mujer,Turista
4,C00007,Mujer,Local
...,...,...,...
2047,C02793,Mujer,Turista
2048,C02794,Mujer,Local
2049,C02796,Mujer,Local
2050,C02798,Mujer,Local


In [6]:
# Crear tabla de dimensiones de tiendas

dim_tienda = df[['id_tienda', 'nombre_tienda', 'ciudad', 'departamento', 'empresa']].drop_duplicates().copy()
dim_tienda['ciudad'] = dim_tienda['ciudad'].fillna('Sin ciudad')
dim_tienda = dim_tienda.drop_duplicates(subset=['id_tienda'], keep='first')
dim_tienda = dim_tienda.sort_values('id_tienda').reset_index(drop=True)

dim_tienda

,id_tienda,nombre_tienda,ciudad,departamento,empresa
0,T001,Koaj Premium Plaza,Medellín,Antioquia,KOAJ
1,T002,Koaj Multicentro,Ibagué,Tolima,KOAJ
2,T003,Koaj Ocean Mall,Santa Marta,Magdalena,KOAJ
3,T004,Koaj Unicentro,Armenia,Quindío,KOAJ
4,T005,Koaj Viva Sincelejo,Sincelejo,Sucre,KOAJ
5,T006,Koaj Único,Pasto,Nariño,KOAJ
6,T007,Koaj San Pedro Plaza,Neiva,Huila,KOAJ
7,T008,Koaj Gran Estación,Bogotá,Bogotá D.C.,KOAJ
8,T009,Koaj Buenavista,Montería,Córdoba,KOAJ
9,T010,Koaj Terraplaza,Popayán,Cauca,KOAJ


In [8]:
# Crear tabla de hechos de ventas

columnas_fact = [
    'id_venta', 'numero_linea', 'fecha_venta',
    'id_cliente', 'id_producto', 'id_tienda',
    'cantidad', 'precio_unitario', 'descuento',
    'metodo_pago', 'canal', 'temporada', 'talla'
]

fact_ventas = df[columnas_fact].copy()

fact_ventas


,id_venta,numero_linea,fecha_venta,id_cliente,id_producto,id_tienda,cantidad,precio_unitario,descuento,metodo_pago,canal,temporada,talla
0,V003565,1,2024-02-04,C00909,P0003,T007,4,179000.0,0.00,Tarjeta Crédito,Tienda Física,Temporada Regular,40
1,V002111,2,2023-01-01,C00348,P0008,T040,1,211000.0,0.00,Efectivo,Tienda Física,Regreso A Clases,M
2,V002185,1,2023-03-31,C00804,P0009,T003,1,128000.0,0.00,Efectivo,Tienda Física,Día Del Padre,Xxl
3,V002828,1,2024-05-13,C00951,P0014,T034,2,133000.0,0.00,Tarjeta Crédito,Tienda Física,Black Friday,38
4,V001693,1,2023-11-14,C01804,P0007,T030,1,61000.0,0.00,Tarjeta Crédito,Tienda Física,Temporada Regular,Xl
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5988,V003749,1,2024-10-19,C01395,P0004,T005,4,180000.0,0.00,Tarjeta Crédito,Tienda Física,Temporada De Diciembre,38
5989,V002277,2,2025-12-04,C02228,P0009,T019,1,130000.0,0.00,Tarjeta Crédito,Tienda Física,Temporada De Diciembre,S
5990,V001587,1,2025-07-11,C01323,P0018,T018,1,77000.0,0.05,Tarjeta Débito,Tienda Física,Temporada Regular,M
5991,V001576,1,2024-01-15,C02669,P0021,T018,4,126000.0,0.05,Efectivo,Tienda Física,Temporada Regular,L


In [9]:
# Guardar las tablas en archivos CSV

fact_ventas.to_csv('../data/processed/fact_ventas.csv', index=False, encoding='utf-8')
dim_producto.to_csv('../data/processed/dim_producto.csv', index=False, encoding='utf-8')
dim_cliente.to_csv('../data/processed/dim_cliente.csv', index=False, encoding='utf-8')
dim_tienda.to_csv('../data/processed/dim_tienda.csv', index=False, encoding='utf-8')

print("Tablas guardadas en data/processed/")

Tablas guardadas en data/processed/
